# 04 - Model Serving & Deployment

**Azure ML MLOps Workshop | Block 3**

### Why Managed Endpoints?

A model in the registry is useful - but value comes from serving predictions.
Azure ML Managed Online Endpoints provide:
- **Autoscaling** infrastructure (no VM management)
- **Blue/green deployments** for safe model updates
- **Built-in auth** and HTTPS
- **Logging** and monitoring integration

### What you'll do:
1. Create a managed online endpoint
2. Deploy the registered model with a scoring script
3. Test with live inspection comments
4. Demonstrate blue/green traffic splitting

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import json

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Create the Online Endpoint

An endpoint is the stable URL. Deployments (model versions) sit behind it.
This separation lets you swap models without changing the client integration.

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

ENDPOINT_NAME = "contoso-lead-classifier"  # <<<< CHANGE THIS - ADD YOUR INITIALS (e.g., contoso-lead-classifier-jd)

# Endpoint names must be unique in the region - append a short suffix if needed
endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Contoso inspection lead classification endpoint",
    auth_mode="key",
    tags={
        "region": "region-1",
        "use_case": "lead-detection",
        "team": "contoso-data-science",
    },
)

endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Endpoint created: {endpoint.name}")
print(f"  Scoring URI: {endpoint.scoring_uri}")
print(f"  State: {endpoint.provisioning_state}")

## Step 2: Create a Deployment (Blue)

The deployment links a registered model + scoring script + environment to the endpoint.

In [ ]:
from azure.ai.ml.entities import (
    ManagedOnlineDeployment,
    CodeConfiguration,
    Environment,
)

MODEL_NAME = "contoso-lead-classifier"

blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=f"azureml:{MODEL_NAME}@latest",
    code_configuration=CodeConfiguration(
        code="../../src/track_a_text",
        scoring_script="score.py",
    ),
    environment=Environment(
        conda_file="../../environment/track_a_text/deployment_env.yml",
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

print("Creating deployment (this takes 5-10 minutes)...")
blue_deployment = ml_client.online_deployments.begin_create_or_update(blue_deployment).result()
print(f"Deployment created: {blue_deployment.name}")

In [ ]:
# Route 100% traffic to the blue deployment
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Traffic routing: 100% -> blue")

## Step 3: Test the Endpoint

Send real inspection comments and get predictions back.
This is the moment where the ML pipeline delivers business value.

In [ ]:
import tempfile
import os

test_comments = [
    "Hydraulic cylinder rod leak detected",
    "Not applicable",
    "Brake pad wear detected, replacement recommended",
    "System functionality test performed, satisfactory result",
    "Operational damage to cylinder, urgent replacement required",
    "Increasing cylinder pipe leak, pipe replacement recommended",
    "Equipment parked at staging area awaiting tires",
]

request_payload = json.dumps({
    "data": [{"comment": c} for c in test_comments]
})

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    f.write(request_payload)
    request_file_path = f.name

response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file=request_file_path,
)

os.remove(request_file_path)

results = json.loads(response)
print("Predictions:")
print("-" * 80)
for r in results["results"]:
    lead = "LEAD" if r["is_lead_opportunity"] else "----"
    print(f"  [{lead}] (prob: {r['probability']:.2f}, conf: {r['confidence']}) {r['comment'][:60]}")

## Step 4: Blue/Green Deployment Pattern

When you have a new model version, you don't replace the existing deployment.
Instead, deploy alongside and gradually shift traffic.

This is how you do safe model updates in production across regions.

In [ ]:
# Hypothetical: deploy a new model version as 'green'
# (In practice, you'd register model v2 from a new experiment first)

# green_deployment = ManagedOnlineDeployment(
#     name="green",
#     endpoint_name=ENDPOINT_NAME,
#     model=f"azureml:{MODEL_NAME}:2",  # model version 2
#     code_configuration=CodeConfiguration(
#         code="../../src/track_a_text",
#         scoring_script="score.py",
#     ),
#     environment=Environment(
#         conda_file="../../environment/track_a_text/deployment_env.yml",
#         image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
#     ),
#     instance_type="Standard_DS2_v2",
#     instance_count=1,
# )
# ml_client.online_deployments.begin_create_or_update(green_deployment).result()

# # Phase 1: Send 10% traffic to green for canary testing
# endpoint.traffic = {"blue": 90, "green": 10}
# ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# # Phase 2: After validation, shift to 50/50
# endpoint.traffic = {"blue": 50, "green": 50}
# ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# # Phase 3: Full cutover to green
# endpoint.traffic = {"blue": 0, "green": 100}
# ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# # Phase 4: Delete the old blue deployment
# ml_client.online_deployments.begin_delete(
#     name="blue", endpoint_name=ENDPOINT_NAME
# ).result()

print("Blue/green pattern (commented out - uncomment when you have model v2):")
print("  1. Deploy green alongside blue")
print("  2. Route 10% traffic to green (canary)")
print("  3. Validate metrics, shift to 50/50")
print("  4. Full cutover to green")
print("  5. Decommission blue")

## Step 5: Check Endpoint Logs

In [ ]:
logs = ml_client.online_deployments.get_logs(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    lines=50,
)
print("Recent deployment logs:")
print(logs)

## Go to Azure ML Studio Now

Navigate to **Endpoints** in the left navigation:

1. Click on **contoso-lead-classifier** endpoint
2. **Details tab** - See the scoring URI, auth keys, traffic split
3. **Test tab** - Paste the JSON payload and test interactively
4. **Logs tab** - View real-time container logs
5. **Monitoring tab** - Request latency, error rates

---

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Managed Endpoints** | No infra management - Azure handles scaling |
| **Scoring Script** | Custom Python that preprocesses and predicts |
| **Blue/Green** | Safe model updates with gradual traffic shifting |
| **Multi-region** | Each region gets its own endpoint; same model registry |
| **Integration** | REST API makes it easy to integrate with existing apps |

---

**Next:** [05 - Model Monitoring](./05_model_monitoring.ipynb)

## Cleanup (run after workshop)

Endpoints incur cost while running. Delete when done.

In [ ]:
# Uncomment to delete the endpoint after the workshop
# ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
# print(f"Endpoint '{ENDPOINT_NAME}' deleted.")